# Generate image based on pin type


> Generate image based on pin type

In [ ]:
#| default_exp image_gen.synthetic_data

In [ ]:
#| hide
%load_ext autoreload 
%autoreload 2

In [ ]:
#| export
from typing import List, Dict, Union
import numpy as np
from pathlib import Path

import inspect
import os
import cv2
from argparse import ArgumentParser
import matplotlib as mpl
from typing import Tuple, List, Dict, Union, TypeVar, NewType
from tqdm.auto import tqdm
import numpy as np
from fastcore.all import *
import shutil
from argparse import ArgumentParser
import random

In [ ]:
#| export
sys.path.append('/home/ai_sintercra/homes/hasan/projects/git_data/cv_tools')
sys.path.append('/home/ai_sintercra/homes/hasan/projects/git_data/2023_easy_pin_detection')

In [ ]:
#| export 
from cv_tools.core import *

In [ ]:
#| exporti
from new_test.pin_map import *
from new_test.core import *

In [ ]:
#| export
OpenCvImage = NewType('OpenCvImage', np.ndarray)

In [ ]:
#| export
def edit_image_old(
        img:OpenCvImage,
        msk:OpenCvImage,
        tmp_img:OpenCvImage,
        recipe_no:str,
        new_part:OpenCvImage,
        msk_part:OpenCvImage,
        ):
    'Change image based on case'
    num_cols_ = split_image_no()[recipe_no]
    pin_map_func = f'pin_map{recipe_no}'

    last_img = img.copy()
    last_msk = msk.copy()
    pin_index_func = globals()[pin_map_func]

    all_cols = []
    num_cols = len(pin_index_func().keys())

    # getting the bounding box coordinates of all columns
    # regarding the recipe no
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
        tst_img=img,
        tmp_img=tmp_img
    )



    for col_no in range(num_cols):

        # for each column get the bounding box coordinates
        col_bbox_coord = col_bbox[col_no]
        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        col_data_msk = msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        

        parts_ = split_image_with_coordinates(
            img=col_data,
            num_splits=num_cols_,
            direction='vertical'
            )

        msk_parts_ = split_image_with_coordinates(
                img=col_data_msk,
                num_splits=num_cols_,
                direction='vertical'
                )
        # getting indexes where pin is there
        pin_indexes = pin_index_func()[col_no]

        all_parts = []
        for (i, part, k),(mi, mpart, mk) in zip(parts_, msk_parts_):
            if recipe_no == '37':
                col_nos = [4,5,6]
            else:
                col_nos = [3,4,5,6]
            if (i not in pin_indexes) \
                and (i not in col_nos):
                # Replace the part with the new part
                # first changing the new_part shape to actual part shape
                try:
                    added_img = create_same_shape(
                                            src_img=part,
                                            dst_img=new_part
                                        )
                    if msk_part is not None:
                        added_img_msk = create_same_shape(
                                                src_img=mpart,
                                                dst_img=msk_part,
                                            )
                    #part = added_img
                    start_x, start_y, end_x, end_y = k
                    center = ((start_x + end_x) // 2, (start_y + end_y) // 2)
                    col_data = seamless_clone(
                        full_img=col_data,
                        replace_part=added_img,
                        center=center
                    )
                    if msk_part is not None:
                        col_data_msk = seamless_clone(
                            full_img=col_data_msk,
                            replace_part=added_img_msk,
                            center=center
                        )
                except Exception as e:
                    print(e)
            #else:
                #part = part

            #all_parts.append(part)

        
        last_img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = col_data
        if msk_part is not None:
            last_msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = col_data_msk


    return last_img, last_msk

In [ ]:
#| export
def augment_image(image, augmentation=None):
    """
    Randomly augment the input image with a 50% chance.
    Possible augmentations: brightness change, noise, blur, or sharpen.
    """
    if random.random() < 0.5:
        return image  # No augmentation 50% of the time
    if augmentation is None:
        augmentation = random.choice(['brightness', 'noise', 'blur', 'sharpen'])
    
    if augmentation == 'brightness':
        factor = random.uniform(0.3, 1.3)
        return cv2.convertScaleAbs(image, alpha=factor, beta=0)
    elif augmentation == 'noise':
        noise = np.random.uniform(0, 1, image.shape).astype(np.uint8)
        return cv2.add(image, noise)
    elif augmentation == 'blur':
        
        return cv2.GaussianBlur(image, (5, 5), 0)
    elif augmentation == 'sharpen':
        kernel = np.array([[-1,-1,-1], [-1,9,-1], [-1,-1,-1]])
        return cv2.filter2D(image, -1, kernel)

In [ ]:
#| export
def edit_image(
        img:OpenCvImage,
        msk:OpenCvImage,
        tmp_img:OpenCvImage,
        recipe_no:str,
        new_part:OpenCvImage,
        msk_part:OpenCvImage,
        ):
    'Change image based on case'
    num_cols_ = split_image_no()[recipe_no]
    pin_map_func = f'pin_map{recipe_no}'

    last_img = img.copy()
    last_msk = msk.copy()
    pin_index_func = globals()[pin_map_func]

    all_cols = []
    num_cols = len(pin_index_func().keys())

    # getting the bounding box coordinates of all columns
    # regarding the recipe no
    bbox_func = globals()[f'get_all_columns_{recipe_no}']
    col_bbox = bbox_func(
        tst_img=img,
        tmp_img=tmp_img
    )


    pin_type = get_pin_type()[recipe_no]
    if '1B' in pin_type:
        col_nos=[1,2,3]
    elif pin_type == '97':
        col_nos = [3,4,5]
    else:
        col_nos=[3,4,5]

    for col_no in range(num_cols):

        # for each column get the bounding box coordinates
        col_bbox_coord = col_bbox[col_no]
        col_data = img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        col_data_msk = msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]]
        

        parts_ = split_image_with_coordinates(
            img=col_data,
            num_splits=num_cols_,
            direction='vertical'
            )

        msk_parts_ = split_image_with_coordinates(
                img=col_data_msk,
                num_splits=num_cols_,
                direction='vertical'
                )
        # getting indexes where pin is there
        pin_indexes = pin_index_func()[col_no]

        all_parts = []
        for (i, part, k),(mi, mpart, mk) in zip(parts_, msk_parts_):
            if (i not in pin_indexes) \
                and (col_no not in col_nos):
                # Replace the part with the new part
                # first changing the new_part shape to actual part shape
                try:
                    added_img = create_same_shape(
                                            src_img=part,
                                            dst_img=new_part
                                        )
                    added_img = augment_image(added_img)
                    if msk_part is not None:
                        added_img_msk = create_same_shape(
                                                src_img=mpart,
                                                dst_img=msk_part,
                                            )
                    #part = added_img
                    start_x, start_y, end_x, end_y = k
                    center = ((start_x + end_x) // 2, (start_y + end_y) // 2)
                    col_data = seamless_clone(
                        full_img=col_data,
                        replace_part=added_img,
                        center=center
                    )
                    if msk_part is not None:
                        col_data_msk = seamless_clone(
                            full_img=col_data_msk,
                            replace_part=added_img_msk,
                            center=center
                        )
                except Exception as e:
                    print(e)
            #else:
                #part = part

            #all_parts.append(part)

        
        last_img[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = col_data
        if msk_part is not None:
            last_msk[col_bbox_coord[0]:col_bbox_coord[1], col_bbox_coord[2]:col_bbox_coord[3]] = col_data_msk


    return last_img, last_msk

In [ ]:
#| hide
im_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/sample_images')

msk_path = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/sample_masks')

TMP_PATH = Path(r'/home/ai_easypid.work/CurrentTrainingData20240812_trn_val/templates')

In [ ]:
#| hide
#| eval:false
missing_pin_im_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_sn_images/one_type')
missing_pin_msk_path = Path(r'/home/ai_sintercra.work/Fail_investigation/Missing_lead/ETPD_WAR_1_02_missing_20240813/missing_pin_sn_masks')
tmp_path = Path(TMP_PATH, '17.png')
tmp_img = read_img(tmp_path)

In [ ]:
sm_fn = Path(im_path).ls()[0]
img = read_img(sm_fn)
name_ = Path(sm_fn).name
msk = read_img(Path(msk_path, name_))
sm_fn

In [ ]:
add_pin_part = read_img(Path(missing_pin_im_path).ls()[0])
show_(add_pin_part)

In [ ]:
for j in ['brightness', 'noise', 'blur', 'sharpen']:
    print(j)
    for i in range(10):
        n_ = augment_image(add_pin_part,augmentation=j )
        show_(n_)

In [ ]:
src_img = sm_fn
recipe_name = get_m_name(f'{src_img}')
new_img,new_msk = edit_image(img, msk, tmp_img, recipe_no=recipe_name, new_part=add_pin_part, msk_part=None)

In [ ]:
show_(new_img)

In [ ]:
#| hide


im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/new_image_source/108').ls()[0]
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/images').ls()[0]
name_  = Path(im_path).name
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/corrected_masks')
msk_path = Path(msk_path, name_)
img = read_img(im_path)
tmp_path = Path(TMP_PATH, '17.png')
tmp_img = read_img(f'{tmp_path}')
new_part_path =Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/sn_img')
new_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/sn_cor_msk')

In [ ]:
#| hide
add_pin_path = Path(new_part_path).ls()[0]
add_msk_path = Path(new_msk_path, Path(add_pin_path).name)
add_pin_part = read_img(add_pin_path)
add_msk_part = read_img(add_msk_path)   

In [ ]:
show_(add_msk_part)

In [ ]:
#| hide
#| eval: false
msk = read_img(msk_path)

In [ ]:
#| hide
#| eval: false
last_img,last_mask = edit_image(
    img,
    msk,
    tmp_img,
    recipe_no='17',
    new_part=add_pin_part,
    msk_part=None


)

In [ ]:
#| hide
show_(last_img)

In [ ]:
#| hide
mpl.rcParams['image.cmap']='gray'

## process to generate image

- get specific recipe 
- save single in a folder
- get image and mask for a recipe (correct mask)
- geneate image for this image.
  - mask will be same for all the images
- may be genearate more image with noise for same image

In [ ]:
#| hide
#| eval: false
DATA_PATH= os.environ.get('DATA_PATH')
TMP_PATH= os.getenv('TEMP_PATH')
TMP_PATH

In [ ]:
#| hide
#| eval: false
add_pin_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/additional_pins/91_des')

In [ ]:
#| export
def edit_image_add_pin_folder(
        img:OpenCvImage,
        tmp_img:OpenCvImage,
        msk:OpenCvImage,
        new_part_folder:Union[str,Path], # loop through each image in this pin folder (each single pin all be added in a image)
        new_msk_part_folder:Union[str,Path]=None, # loop thorugh each mask fodler but name will be new_part folder, if it is None, then just empty mask will be created
        recipe_no:str='17',
        show:bool=False,
        gen_im_path:Union[Path, str]=None,
        gen_msk_path:Union[Path, str]=None,
        fn=None # when loop through each image, the name of main image will be same, to make sure different images are created, idx is added inside name
    ):
    'Create image based from all pins insider new_part_folder'
    try:
        new_part_folder = Path(new_part_folder)
        new_part_fns = new_part_folder.ls()
    except Exception as e:
        print(e)
    if len(new_part_fns) >=1:
        for idx, i in tqdm(enumerate(new_part_fns), total=len(new_part_fns)):
            name_= Path(i).name
            new_part = read_img(i, cv=True)
            if new_msk_part_folder is None:
                new_msk_part = np.zeros_like(new_part)
            else:
                new_msk_part = read_img(Path(new_msk_part_folder, name_), cv=True)
            new_img, new_msk = edit_image(
                img=img, 
                msk=msk,
                tmp_img=tmp_img,
                recipe_no=recipe_no,
                new_part=new_part,
                msk_part=new_msk_part
            )
            if show:
                show_(new_img)
            if gen_im_path is not None:
                # create folder if its not there
                Path(gen_im_path).mkdir(exist_ok=True, parents=True)

                cv2.imwrite(f'{gen_im_path}/{fn}_gen_image_{idx}_{name_}',new_img)
            if gen_msk_path is not None:
                # create folder if its not there
                Path(gen_msk_path).mkdir(exist_ok=True, parents=True)
                cv2.imwrite(f'{gen_msk_path}/{fn}_gen_image_{idx}_{name_}',new_msk)




In [ ]:
#| hide
#| eval: false
#model_path = Path(DATA_PATH, 'easy_endline/models/time_11_18_24_val_frGrnd0.9470_epoch_185.h5')
#data_path = Path(DATA_PATH, 'easy_endline/Fail_upto20240206_des/generated_src')
#data_path.mkdir(exist_ok=True, parents=True)
#sn_pin_path = Path(DATA_PATH, 'easy_endline/Fail_upto20240206_des/additional_pins/').ls()[2]
#im_path = Path(data_path, 'images').ls()[0]
#msk_path = Path(data_path, 'masks/processed_mask').ls()[0]
tmp_path = Path(os.getenv('TEMP_PATH'))

im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/images')
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/corrected_masks')
add_pin_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new_model/time_16_49_30_val_frGrnd0.9569_epoch_159_additional_pins/additional_pins')
gen_im_path = Path(Path(add_pin_path).parent, 'gen_images_test')
gen_msk_path = Path(Path(add_pin_path).parent, 'gen_masks_test')

In [ ]:

edit_image_add_pin_folder(
        img=read_img(im_path.ls()[0]),
        tmp_img=read_img(f'{tmp_path}/87.png'),
        msk=read_img(msk_path.ls()[0]),
        new_part_folder=add_pin_path,
        recipe_no='87',
        gen_im_path=gen_im_path,
        gen_msk_path=gen_msk_path
    )

In [ ]:
#| hide
#| eval: false
im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/images')
msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/corrected_masks')
add_pin_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/sn_img')
add_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/sn_cor_msk')
gen_im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/gen_images')
gen_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/main_im2_cropped_masks_new/time_13_02_08_val_frGrnd0.9557_epoch_180_missing_pins/images_masks_new/time_11_18_24_val_frGrnd0.9470_epoch_185_additional_pins/gen_masks')

In [ ]:
#| hide
#| eval: false
img = read_img(im_path.ls()[0])
name_ = Path(im_path.ls()[0]).name
msk = read_img(Path(im_path, name_))
tmp_img = read_img(tmp_path)

In [ ]:
#| hide
s_ap = add_pin_path.ls()[1]
s_ap

In [ ]:
#| export
def generate_image_from_path(
        im_path:Union[Path,str],
        msk_path:Union[Path,str],
        tmp_path:Union[Path,str],
        add_pin_path:Union[Path,str],
        add_msk_path:Union[Path,str]=None,
        gen_im_path:Union[Path, str]=None,
        gen_msk_path:Union[Path, str]=None,
      ):
        'Generate images based on failed imges'

        im_path = Path(im_path)
        for idx, i in enumerate(im_path.ls()):
                recipe_no = get_m_name(f'{i}')
                name_ = Path(i).name
                img = read_img(i, cv=True)
                try:
                        msk = read_img(f'{msk_path}/{name_}', cv=True)
                except Exception as e:
                        print(e)
                        print('No mask found')

                tmp_img = read_img(f'{tmp_path}/{recipe_no}.png', cv=True)

                edit_image_add_pin_folder(
                        img=img,
                        tmp_img=tmp_img,
                        msk=msk,
                        new_part_folder=add_pin_path,
                        new_msk_part_folder=add_msk_path,
                        recipe_no=recipe_no,
                        show=False,
                        gen_im_path=gen_im_path,
                        gen_msk_path=gen_msk_path,
                        fn=f'src_img_recipe_{recipe_no}_idx_{idx}',
                )




In [ ]:
#| hide
#recipe_no='17'
#im_path = Path(fr'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/new_image_source/{recipe_no}')
##msk_path = Path(fr'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/new_image_source/{recipe_no}_processed_masks')
#mks_path=None
#add_pin_path = Path(fr'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/additional_pins/{recipe_no}_des')
#tmp_path = Path(TMP_PATH,f'{recipe_no}.png')
#gen_im_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/generated_des/images')
##gen_msk_path = Path(r'/home/user/Schreibtisch/projects/data/easy_endline/Fail_upto20240206_des/generated_des/masks')
#gen_msk_path = None

In [ ]:
#| export
def parse_args_():
    parser = ArgumentParser()
    parser.add_argument(
        "--im_path",
        type=str,
        required=True,
        help="Path to images, this image will be used to generate new images[this image + addtional pins]"
    )
    parser.add_argument(
        "--msk_path",
        type=str,
        default=None,
        help="Path to masks, this mask will be used to generate new masks"
    )
    parser.add_argument(
        "--tmp_path",
        type=str,
        required=True,
        help="Path to tmp image, this is the template image"
    )
    parser.add_argument(
        "--add_pin_path",
        type=str,
        required=True,
        help="Path to additional pins, this will be used to generate new images"
    )
    parser.add_argument(
        "--add_msk_path",
        default=None,
        help="In case of mask is available, missing pin case, then mask will be created, otherwise empty masks will be created "
    )
    parser.add_argument(
        "--gen_im_path",
        type=str,
        default=None,
        required=False,
        help="Path to saved generated images"
    )

    parser.add_argument(
        "--gen_msk_path",
        type=str,
        required=False,
        default=None,
        help="Path to saved generated masks"
    )
    parser.add_argument(
        "--show",
        type=bool,
        required=False,
        help="Show generated images if True"
    )
    return parser.parse_args()

In [ ]:
from pathlib import Path
from cv_tools.imports import *
from cv_tools.core import *

In [ ]:
#| hide
#| eval: false
im_path = Path(r'E:\CurrentTrainingData20240812_trn_val\sample_images')
msk_dest = Path(r'E:\CurrentTrainingData20240812_trn_val\sample_masks')
msk_path = Path(r'E:\CurrentTrainingData20240418_trn_val\sample_masks')
im_names = get_name_(im_path.ls())

In [ ]:
msk_path.ls()

In [ ]:
masks = list(filter(lambda x: Path(x).name in im_names, msk_path.ls()))
len(masks)

In [ ]:
masks

In [ ]:
msk_dest

In [ ]:
for i in msk_path.ls():
    if Path(i).name in get_name_(masks):
        shutil.copyfile(i, Path(msk_dest, Path(i).name))
        print(i)

In [ ]:
#| export
def main_():
    args = parse_args_()
    generate_image_from_path(
        im_path=Path(args.im_path),
        msk_path=Path(args.msk_path),
        tmp_path=Path(args.tmp_path),
        add_pin_path=Path(args.add_pin_path),
        add_msk_path=args.add_msk_path,
        gen_im_path=Path(args.gen_im_path),
        gen_msk_path=Path(args.gen_msk_path)
    )


In [ ]:
# So checking generated image has masks or not

In [ ]:
#| export
if __name__ == '__main__':
    main_()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export('05_generate_image.ipynb')